# Qwen LoRA SFT / Data Pipeline / DPO Learning Notebook

这个 notebook 是本项目的“可操作学习地图”。它把命令行脚本串成可以逐格运行的流程，帮助理解：

- Qwen base 推理是什么样子。
- LoRA SFT 怎样准备数据、训练 adapter、加载 adapter。
- 为什么公开通用数据集能跑通工程闭环，但不一定修正 LoRA/SFT/DPO 技术概念误解。
- Stage 2B 如何做自采集数据、清洗、转换。
- Stage 3B 如何训练 custom LoRA SFT。
- Stage 4B 如何做 base vs public-SFT vs custom-SFT 三方对比。
- DPO 为什么更吃显存，以及 8GB 显存下如何保守测试。

当前 notebook 覆盖到 Stage 4B。Stage 5 DPO 部分先保留学习骨架，后续边做边补。


## 0. 使用方式

建议用 `qwen-lora-local` 环境打开这个 notebook。大多数 cell 是轻量检查；训练 cell 默认不会执行，需要手动把开关改成 `True`。

如果你只是复盘项目，按顺序运行环境检查、推理、报告读取即可。如果你要重新训练，再打开训练开关。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

os.environ["HF_HOME"] = str((PROJECT_ROOT / ".hf_cache").resolve())
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("HF_HOME:", os.environ["HF_HOME"])

def run_cmd(args, timeout=None):
    print("$", " ".join(str(x) for x in args))
    result = subprocess.run(
        [str(x) for x in args],
        cwd=PROJECT_ROOT,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        timeout=timeout,
    )
    print(result.stdout)
    if result.stderr:
        print("[stderr]")
        print(result.stderr)
    result.check_returncode()
    return result

## 1. 环境检查

先确认 Python 包版本和 CUDA。这个项目之前在 Windows 原生环境里遇到过高版本 Hugging Face 栈导致 `python.exe` 进程级崩溃的问题，所以环境版本是项目经验的一部分。

In [ ]:
run_cmd([sys.executable, "scripts/check_env.py"], timeout=60)
run_cmd([
    sys.executable,
    "-c",
    "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')",
], timeout=60)

## 2. 先讲清楚术语

- SFT 是 supervised fine-tuning，指训练目标和数据形式。
- LoRA 是 parameter-efficient fine-tuning 方法，指只训练 adapter 小矩阵。
- LoRA SFT 的意思是：用 LoRA adapter 来做 SFT。
- DPO 是 direct preference optimization，使用 chosen/rejected 偏好对进一步调模型行为。

Stage 4A 的重要发现是：公开通用数据集虽然跑通了 LoRA SFT，但没有修正这些技术概念误解。

## 3. Base 模型推理

先看没有 adapter 的 Qwen 怎么回答。这是后续对比的 baseline。

In [ ]:
run_cmd([
    sys.executable,
    "scripts/infer.py",
    "--prompt", "请用三点解释什么是LoRA微调。",
    "--max_new_tokens", "96",
    "--local_files_only",
], timeout=180)

## 4. Stage 2A：公开数据集准备

当前公开数据集已经准备好：

- raw: `data/raw/alpaca_gpt4_data_zh_1200.jsonl`
- train: `data/processed/sft_train.jsonl`
- eval: `data/processed/sft_eval.jsonl`

如果以后要重跑，把下面的 `RUN_PUBLIC_DATA_PREP` 改成 `True`。

In [ ]:
RUN_PUBLIC_DATA_PREP = False

if RUN_PUBLIC_DATA_PREP:
    run_cmd([
        sys.executable,
        "scripts/download_hf_sft_data.py",
        "--dataset_name", "llm-wizard/alpaca-gpt4-data-zh",
        "--split", "train",
        "--output_file", "data/raw/alpaca_gpt4_data_zh_1200.jsonl",
        "--max_samples", "1200",
        "--seed", "42",
    ], timeout=600)
    run_cmd([
        sys.executable,
        "scripts/prepare_data.py",
        "--input_file", "data/raw/alpaca_gpt4_data_zh_1200.jsonl",
        "--train_file", "data/processed/sft_train.jsonl",
        "--eval_file", "data/processed/sft_eval.jsonl",
        "--eval_ratio", "0.1",
        "--max_samples", "1200",
        "--min_answer_chars", "20",
        "--seed", "42",
    ], timeout=120)
else:
    for path in ["data/processed/sft_train.jsonl", "data/processed/sft_eval.jsonl"]:
        rows = [line for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
        print(path, "rows =", len(rows))

## 5. Stage 3A：公开数据 LoRA SFT

这一步训练耗时约 24 分钟，训练时显存观察约 `5.5GB / 8GB`。默认不执行。

已经完成的 adapter：`outputs/sft_lora_qwen05b_public`。

In [ ]:
RUN_PUBLIC_SFT_TRAINING = False

if RUN_PUBLIC_SFT_TRAINING:
    run_cmd([
        sys.executable,
        "scripts/train_sft_lora.py",
        "--train_file", "data/processed/sft_train.jsonl",
        "--eval_file", "data/processed/sft_eval.jsonl",
        "--output_dir", "outputs/sft_lora_qwen05b_public",
        "--max_length", "512",
        "--batch_size", "1",
        "--grad_accum", "8",
        "--epochs", "1",
        "--logging_steps", "10",
        "--eval_steps", "100",
        "--save_steps", "100",
        "--report_to", "none",
        "--local_files_only",
    ], timeout=7200)
else:
    adapter_dir = Path("outputs/sft_lora_qwen05b_public")
    print("Adapter exists:", adapter_dir.exists())
    print("Adapter files:", [p.name for p in adapter_dir.glob("adapter_*")])

## 6. Stage 4A：Base vs Public-SFT 对比

这一步是现在最重要的结论来源：公开数据 SFT 能训练成功，但没有修正 LoRA/SFT/DPO 的概念误解。

In [ ]:
RUN_STAGE4A_COMPARE = False

if RUN_STAGE4A_COMPARE:
    run_cmd([
        sys.executable,
        "scripts/compare_outputs.py",
        "--prompt_file", "data/samples/smoke_prompts.jsonl",
        "--adapter_path", "outputs/sft_lora_qwen05b_public",
        "--output_file", "reports/compare_outputs_public_sft.jsonl",
        "--max_new_tokens", "96",
        "--local_files_only",
    ], timeout=900)

rows = [json.loads(line) for line in Path("reports/compare_outputs_public_sft.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
for row in rows:
    print("=" * 80)
    print("Prompt:", row["prompt"])
    print("\nBase:\n", row["base_answer"][:500])
    print("\nPublic-SFT:\n", row["sft_answer"][:500])

## 7. Stage 4A 结论

公开通用数据集的价值：

- 证明了数据转换、tokenizer、LoRA 训练、adapter 保存/加载都能工作。
- 得到了一个可比较的 public-SFT baseline。

公开通用数据集的不足：

- 没有修正 LoRA/SFT/DPO 的目标技术概念。
- LoRA 仍可能被解释成通信 LoRa。
- SFT/DPO 仍可能被解释成无关缩写。

所以 Stage 2B 不是锦上添花，而是必要步骤：我们需要自采集技术数据来教模型目标概念和项目叙述方式。

## 8. Stage 2B：自采集技术数据

Stage 2B 已经完成并根据 Stage 3B 训练反馈修订过。第一版生成 160 条 seed 样本，但训练后仍出现幻觉，说明数据里泛化的项目总结样本太多。

当前版本做了两件事：

- 把 `project_record_summary` 从 60 条降到 20 条，减少模板化项目记录对模型的干扰。
- 新增 12 条 targeted QA，直接覆盖 LoRA/SFT/DPO、public-SFT 失败原因、数据清洗、DPO 显存、loss vs behavior 等固定 prompt。

当前产物：

- `data/raw/custom_sources.jsonl`：10 个项目自有技术来源。
- `data/raw/custom_cleaned_chunks.jsonl`：85 个清洗后 chunk。
- `data/raw/custom_instruction_seed.jsonl`：132 条 instruction-answer seed。
- `data/processed/custom_sft_train.jsonl`：119 条 train。
- `data/processed/custom_sft_eval.jsonl`：13 条 eval。


In [ ]:
RUN_STAGE2B_PREP = False

if RUN_STAGE2B_PREP:
    run_cmd([
        sys.executable,
        "scripts/prepare_custom_technical_data.py",
        "--raw_sources_file", "data/raw/custom_sources.jsonl",
        "--cleaned_chunks_file", "data/raw/custom_cleaned_chunks.jsonl",
        "--instruction_seed_file", "data/raw/custom_instruction_seed.jsonl",
        "--train_file", "data/processed/custom_sft_train.jsonl",
        "--eval_file", "data/processed/custom_sft_eval.jsonl",
        "--eval_ratio", "0.1",
        "--max_doc_samples", "20",
        "--seed", "42",
    ], timeout=300)

for path in [
    "data/raw/custom_sources.jsonl",
    "data/raw/custom_cleaned_chunks.jsonl",
    "data/raw/custom_instruction_seed.jsonl",
    "data/processed/custom_sft_train.jsonl",
    "data/processed/custom_sft_eval.jsonl",
]:
    rows = [line for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    print(path, "rows =", len(rows))


## 9. Stage 2B 校验

Stage 2B 的校验重点：

- train/eval 都是 Qwen chat JSONL。
- role 顺序是 `system -> user -> assistant`。
- prompt 不重复。
- 原始 token 长度不超过 512，避免训练时截断。
- 能被 `ChatSFTDataset` 正常编码。

In [ ]:
for path in ["data/processed/custom_sft_train.jsonl", "data/processed/custom_sft_eval.jsonl"]:
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    roles_ok = all([m["role"] for m in row["messages"]] == ["system", "user", "assistant"] for row in rows)
    prompts = [row["messages"][1]["content"] for row in rows]
    print(path)
    print("rows:", len(rows))
    print("roles_ok:", roles_ok)
    print("unique_prompts:", len(set(prompts)))
    print("sample prompt:", prompts[0][:120].replace("\n", " "))
    print("sample answer:", rows[0]["messages"][2]["content"][:160].replace("\n", " "))
    print()

## 10. Stage 3B：custom-data LoRA SFT

Stage 3B 已经完成，输出 adapter：

```text
outputs/sft_lora_qwen05b_custom
```

训练结果摘要：

- Train rows: 119
- Eval rows: 13
- Trainable params: 4,399,104，约 0.8826%
- Runtime: 约 12.3 分钟
- Final train loss: 0.4656
- Best observed eval loss: 0.8311，约在 epoch 5

一个很重要的观察：10 epoch 最终能让 target behavior 更明显，但 eval loss 在 epoch 5 后略微上升，说明小数据下有过拟合风险。以后可以考虑保存/选择 epoch 5 附近的最佳 checkpoint。

默认不执行训练；如果你要复跑，把 `RUN_CUSTOM_SFT_TRAINING` 改成 `True`。


In [ ]:
RUN_CUSTOM_SFT_TRAINING = False

if RUN_CUSTOM_SFT_TRAINING:
    run_cmd([
        sys.executable,
        "scripts/train_sft_lora.py",
        "--train_file", "data/processed/custom_sft_train.jsonl",
        "--eval_file", "data/processed/custom_sft_eval.jsonl",
        "--output_dir", "outputs/sft_lora_qwen05b_custom",
        "--max_length", "512",
        "--batch_size", "1",
        "--grad_accum", "4",
        "--epochs", "10",
        "--logging_steps", "10",
        "--eval_steps", "50",
        "--save_steps", "50",
        "--report_to", "none",
        "--local_files_only",
    ], timeout=7200)
else:
    print("Custom SFT training is skipped. Set RUN_CUSTOM_SFT_TRAINING=True when ready.")
    print("Current local output: outputs/sft_lora_qwen05b_custom")
    print("Report: reports/stage3b_custom_lora_sft_report.md")


## 11. Stage 4B：Base vs Public-SFT vs Custom-SFT 三方对比

Stage 4B 已经完成。对比文件：

```text
reports/compare_outputs_three_way_custom.jsonl
reports/stage4b_three_way_comparison_report.md
```

结果不是“全胜”，而是更真实也更有价值：custom-SFT 明显改善 6/8 个固定技术 prompt，同时留下 2 个 badcase，刚好能指导下一轮 Stage 2B.2 小数据补丁。

已明显改善的方向：

- LoRA 不再解释成无线通信 LoRa。
- SFT 和 LoRA 的关系解释正确。
- DPO 和 SFT 的区别解释正确。
- 自采集数据清洗流程更贴合项目。
- 8GB DPO 显存风险和降显存手段更贴合本机。
- 面试数据管线叙述更完整。

仍需补数据的方向：

- 为什么 public-SFT 没修正概念，反而说明 Stage 2B 有必要。
- 为什么不能只看 loss 判断 SFT 是否成功。


In [ ]:
RUN_STAGE4B_COMPARE = False

if RUN_STAGE4B_COMPARE:
    run_cmd([
        sys.executable,
        "scripts/compare_three_outputs.py",
        "--prompt_file", "data/samples/custom_technical_prompts.jsonl",
        "--public_adapter_path", "outputs/sft_lora_qwen05b_public",
        "--custom_adapter_path", "outputs/sft_lora_qwen05b_custom",
        "--output_file", "reports/compare_outputs_three_way_custom.jsonl",
        "--max_new_tokens", "128",
        "--temperature", "0",
        "--local_files_only",
    ], timeout=7200)
else:
    print("Stage 4B comparison is skipped. Set RUN_STAGE4B_COMPARE=True to rerun.")

comparison_path = Path("reports/compare_outputs_three_way_custom.jsonl")
if comparison_path.exists():
    rows = [json.loads(line) for line in comparison_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("comparison rows:", len(rows))
    for i, row in enumerate(rows, start=1):
        print(f"{i}. prompt:", row["prompt"][:80])
        print("   custom:", row["custom_sft_answer"][:120].replace("\n", " "))


## 12. DPO 和显存预判

当前观察：

- LoRA SFT 训练约 `5.5GB / 8GB`。
- Adapter 推理约 `1.2GB / 8GB`。
- Stage 3B custom LoRA SFT 也能在 8GB 上完成。

DPO 会更吃显存，因为它通常需要 chosen/rejected 两个回答，并且要和 reference policy 对比。8GB 不是完全不能做，但第一版必须非常保守：小模型、LoRA、batch size 1、短序列、少量 pair、少 eval，并尽量共享 reference。


In [ ]:
print(Path("reports/vram_and_dpo_plan.md").read_text(encoding="utf-8")[:2000])

## 13. 当前验收清单

- Stage 0/1：环境和 base 推理完成。
- Stage 2A：公开中文 SFT 数据完成。
- Stage 3A：public LoRA SFT 完成。
- Stage 4A：base vs public-SFT 对比完成，结论是没有修正目标技术概念。
- Stage 2B：自采集技术数据完成并修订，当前 119 train / 13 eval。
- Stage 3B：custom-data LoRA SFT 完成，输出 `outputs/sft_lora_qwen05b_custom`。
- Stage 4B：三方对比完成，custom-SFT 改善 6/8 个固定技术 prompt。
- 下一步：先检查和理解结果；如果继续推进，做 Stage 2B.2 小数据补丁，再进入 tiny DPO smoke test。

后续每完成一个阶段，就继续更新这个 notebook。这个文件的作用是把项目从“命令集合”变成一条能逐格运行、逐步理解的学习路线。
